In [ ]:
# Cell 1: Load Quora Dataset
import pandas as pd
import re
import os

# Tumhara specific path
file_path = "C:/Users/Z/plagiarism-detection/data/raw/quora/questions.csv"

# Check if file exists
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"✅ Loaded successfully!")
    print(f"Shape: {df.shape}")
else:
    print(f"❌ File not found at: {file_path}")
    # Try alternative names
    possible_files = ["questions.csv", "quora.csv", "train.csv", "quora_questions.csv"]
    for f in possible_files:
        alt_path = f"C:/Users/Z/plagiarism-detection/data/raw/quora/{f}"
        if os.path.exists(alt_path):
            df = pd.read_csv(alt_path)
            print(f"✅ Loaded from: {f}")
            break

In [ ]:
# Pehle 5 rows dekho
print("First 5 rows:")
print(df.head())
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Check konsa column kaunsa hai
for col in df.columns:
    print(f"{col}: {df[col].iloc[0][:50] if df[col].dtype == 'object' else df[col].iloc[0]}")

In [ ]:
# Cell 4: Clean Function
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
# Cell 5: Apply Cleaning (Corrected column names)
import re

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Remove missing values
initial_len = len(df)
df = df.dropna()
print(f"Dropped {initial_len - len(df)} rows with missing values")

# Clean questions (correct column names: question1, question2)
df['q1_clean'] = df['question1'].apply(clean_text)
df['q2_clean'] = df['question2'].apply(clean_text)

print("✅ Cleaning done!")
print(f"\nSample after cleaning:")
print(df[['question1', 'q1_clean', 'question2', 'q2_clean', 'is_duplicate']].head())

In [ ]:
# Cell 6: Remove short questions
df = df[df['q1_clean'].str.split().str.len() >= 3]
df = df[df['q2_clean'].str.split().str.len() >= 3]

print(f"After removing short questions: {len(df)} rows")

In [ ]:
# Cell 7: Train-Test Split
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['is_duplicate']
)

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")
print(f"\nTrain class distribution:")
print(train_df['is_duplicate'].value_counts(normalize=True))

In [ ]:
# Complete path use karo
import pandas as pd
import os

# Tumhara full path
save_path = "C:/Users/Z/plagiarism-detection/data/processed/"

# Folder create karo
os.makedirs(save_path, exist_ok=True)

# Save karo
train_df[['q1_clean', 'q2_clean', 'is_duplicate']].to_csv(save_path + "quora_train.csv", index=False)
test_df[['q1_clean', 'q2_clean', 'is_duplicate']].to_csv(save_path + "quora_test.csv", index=False)

print(f"✅ Saved to: {save_path}")
print("Files created:")
print(os.listdir(save_path))